# nn-module-subclass — ex2: diagnose missing super().__init__()

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `nn-module-subclass`. Running the final beacon cell reports progress against the `PyTorch: nn.Module subclassing` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: nn.Module subclassing` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`nn-module-subclass`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "nn-module-subclass"
DD_SUBTOPIC = "PyTorch: nn.Module subclassing"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `nn.Module` subclassing — quick refresher

Every learnable building block in PyTorch is an `nn.Module` subclass. The minimal pattern is:

```python
class MyLayer(nn.Module):
    def __init__(self, ...):
        super().__init__()      # MUST be first — wires up _parameters / _modules dicts
        self.weight = nn.Parameter(...)
    def forward(self, x):
        return ...              # never call .forward() directly — use module(x)
```

**Two non-obvious rules.**
1. If `__init__` does anything (assigns Parameters, sub-Modules, or buffers), it MUST call `super().__init__()` first. Forgetting this raises `AttributeError: cannot assign parameter before Module.__init__() call`.
2. Modules with no state can omit `__init__` entirely and just define `forward` (e.g. ARENA's `ReLU`). The base `nn.Module.__init__` runs implicitly.

**Call convention.** Use `module(x)`, never `module.forward(x)` — the `__call__` wrapper runs hooks (pre/post forward, gradient hooks) that you lose by calling forward directly.

### Exercise 2 — diagnose missing super().__init__()

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Analyze
> LO: Diagnose the AttributeError raised when an nn.Module subclass with state forgets to call super().__init__(), then fix it.
> Keywords: super, __init__, debugging, AttributeError
> ```

**KCs targeted:** `module-init-super-call`

Implement `ex2_fix_broken_scaler(scale_value)`. The exercise has two parts:

**Part A — observe the bug.** A buggy class `BrokenScaler` is defined for you below the test cell (in the solution). It has `self.scale = nn.Parameter(...)` in `__init__` but forgets to call `super().__init__()` first. Attempting to instantiate it raises `AttributeError: cannot assign parameter before Module.__init__() call`.

**Part B — write the fix.** Define a class `FixedScaler` that:
1. Subclasses `t.nn.Module`.
2. In `__init__(self, scale_value)` calls `super().__init__()` FIRST.
3. Stores `self.scale = nn.Parameter(t.tensor(float(scale_value)))`.
4. `forward(self, x)` returns `x * self.scale`.

Return an INSTANCE of `FixedScaler` initialized with `scale_value`.

The test asserts: (a) the broken version really does raise, (b) your fixed version constructs cleanly, (c) the parameter is registered (`.parameters()` is non-empty), (d) forward works.

In [ ]:
def ex2_fix_broken_scaler(scale_value: float):
    class FixedScaler(t.nn.Module):
        def __init__(self, scale_value):
            super().__init__()
            self.scale = t.nn.Parameter(t.tensor(float(scale_value)))
        def forward(self, x):
            return x * self.scale
    return FixedScaler(scale_value)


<details><summary>Solution</summary>

```python
def ex2_fix_broken_scaler(scale_value: float):
    class FixedScaler(t.nn.Module):
        def __init__(self, scale_value):
            super().__init__()
            self.scale = t.nn.Parameter(t.tensor(float(scale_value)))
        def forward(self, x):
            return x * self.scale
    return FixedScaler(scale_value)
```

**Why the error message points at Module.__init__.** The `nn.Module.__setattr__` override intercepts every attribute assignment on a Module to check whether the value is a `Parameter` / `Module` / `Buffer` and route it to the right internal dict. But those dicts (`_parameters`, `_modules`, `_buffers`) don't exist until `Module.__init__` creates them. So the first `self.scale = nn.Parameter(...)` assignment can't find `self._parameters` to write into → AttributeError.

**Why this only fails for stateful Modules.** The `SquareLayer` from ex1 had no attribute assignments in its (absent) `__init__`, so the missing super() call never bit. The bug ONLY surfaces when you assign a Parameter / Module / Buffer attribute.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()